# C2S Frozen Embedding + Logistic Regression Baseline

This notebook answers: **what can Cell2Sentence already do, out of the box, without any fine-tuning?**

| Stage | Details |
|---|---|
| Encoder | C2S-Pythia-410m, **frozen** (no gradient updates) |
| Embedding | Last-token hidden state, extracted in one forward pass per batch |
| Classifier | `LogisticRegression(class_weight='balanced')` fitted on embeddings |
| Training data | Same 8 organs as v6 (same preprocessing, same MIN/MAX_CELLS_PER_TYPE) |
| Inference | Lab validation datasets — predict which training class each lab cell maps to |

**What this is not**: fine-tuning. The C2S weights never change. This is a frozen linear probe.

**Interpretation**: if this already performs well, HCE fine-tuning is adding less incremental value. If it performs poorly, fine-tuning is essential.

## 1. Imports

In [ ]:
import os, sys, warnings, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

import scanpy as sc
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import matplotlib.pyplot as plt
import scipy.sparse as sp
import h5py

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
from cell2sentence.hierarchy_utils import deduplicate_hierarchy

warnings.filterwarnings('ignore')
print('=' * 60)
print('  IMPORTS OK')
print(f'  PyTorch  : {torch.__version__}')
print(f'  CUDA     : {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}')
print('=' * 60)

## 2. Configuration

In [ ]:
OUT_DIR        = 'multi_tissue_lr_baseline_results'
C2S_MODEL_NAME = 'vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation'

# ── Training organs (same as v6) ──────────────────────────────────────────────
ORGAN_CONFIGS = [
    {'name': 'lung',         'path': 'lung.h5ad',                       'label_col': 'ann_finest_level', 'gene_col': 'feature_name', 'hierarchy_cols': ['ann_level_1','ann_level_2','ann_level_3','ann_level_4','ann_level_5'], 'coarse_col': None,              'id': 0},
    {'name': 'brain_glia',   'path': 'brain_new.h5ad',                  'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],             'coarse_col': 'supercluster_term','id': 1},
    {'name': 'brain_neurons','path': 'brain_neurons_processed.h5ad',    'label_col': 'label',            'gene_col': 'feature_name', 'hierarchy_cols': [],             'coarse_col': None,              'id': 2},
    {'name': 'liver',        'path': 'census_data/liver.h5ad',          'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],             'coarse_col': None,              'id': 3},
    {'name': 'lymph_node',   'path': 'census_data/lymph_node.h5ad',     'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],             'coarse_col': None,              'id': 4},
    {'name': 'bone_marrow',  'path': 'census_data/bone_marrow.h5ad',    'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],             'coarse_col': None,              'id': 5},
    {'name': 'lymphoid',     'path': 'lab-data/lymphoid.h5ad',          'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],             'coarse_col': None,              'id': 6},
    {'name': 'myeloid',      'path': 'lab-data/myeloid.h5ad',           'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],             'coarse_col': None,              'id': 7},
]

# ── Lab validation datasets (zero-shot inference targets) ─────────────────────
LAB_CONFIGS = [
    {'name': 'All_cells (glioma)', 'path': 'All_cells.h5ad',                 'label_col': 'predicted.high_hierarchy'},
    {'name': 'Brain_normal',       'path': 'lab-data/Brain_normal.h5ad',      'label_col': 'cell_type'},
    {'name': 'Liver_normal',       'path': 'lab-data/Liver_normal.h5ad',      'label_col': 'cell_type'},
    {'name': 'Lymph_node_normal',  'path': 'lab-data/Lymph_node_normal.h5ad', 'label_col': 'cell_type'},
]

# ── Hyperparameters ───────────────────────────────────────────────────────────
TOP_K_GENES        = 200
MAX_CELLS_PER_TYPE = 300
MIN_CELLS_PER_TYPE = 100
BATCH_SIZE         = 32   # larger batch OK since no gradient
MAX_SEQ_LEN        = 512
SEED               = 42

# Logistic regression settings
LR_MAX_ITER = 1000
LR_C        = 1.0   # inverse regularization strength

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'  Device : {device}  |  Output : {OUT_DIR}')
print('[OK] Config ready')

## 3. Data Loading & Preprocessing
*(Same as v6 — same organs, same synonym map, same deduplication)*

In [ ]:
def is_valid(value):
    if pd.isna(value): return False
    return str(value).strip().lower() not in ('', 'nan', 'none', 'unknown', 'na', 'n/a')

def get_gene_symbols(adata):
    if 'feature_name' in adata.var.columns:
        return np.array(adata.var['feature_name'].astype(str))
    return np.array(adata.var_names.astype(str))

def load_organ(cfg):
    name, label_col = cfg['name'], cfg['label_col']
    t0 = time.time()
    adata = sc.read_h5ad(cfg['path'], backed='r')
    gene_symbols = get_gene_symbols(adata)
    valid_mask = adata.obs[label_col].apply(is_valid)
    obs        = adata.obs[valid_mask].copy()
    valid_idx  = np.where(valid_mask.values)[0]
    labels  = obs[label_col].astype(str).values
    sampled = []
    for lbl in np.unique(labels):
        pos = np.where(labels == lbl)[0]
        if len(pos) > MAX_CELLS_PER_TYPE:
            pos = np.random.choice(pos, MAX_CELLS_PER_TYPE, replace=False)
        sampled.extend(pos.tolist())
    sampled   = np.sort(np.array(sampled, dtype=np.int64))
    obs       = obs.iloc[sampled].copy()
    valid_idx = valid_idx[sampled]
    counts = obs[label_col].value_counts()
    keep   = counts[counts >= MIN_CELLS_PER_TYPE].index
    mask      = obs[label_col].isin(keep)
    obs       = obs[mask].copy()
    valid_idx = valid_idx[mask.values]
    print(f'  [{name}] {len(obs):,} cells | {obs[label_col].nunique()} types | {time.time()-t0:.1f}s')
    return adata, obs, valid_idx, gene_symbols

def cell_to_text_backed(h5_path, row_indices, gene_symbols, top_k=200, desc='cells'):
    texts = []
    with h5py.File(h5_path, 'r') as f:
        X = f['X']
        indptr = X['indptr'][:]
        indices_ds, data_ds = X['indices'], X['data']
        for row_idx in tqdm(row_indices, desc=f'  {desc}', leave=False):
            start, end = int(indptr[row_idx]), int(indptr[row_idx + 1])
            if start == end:
                texts.append('')
                continue
            vals = data_ds[start:end]
            cols = indices_ds[start:end]
            order = (np.argsort(vals)[::-1] if len(vals) <= top_k
                     else np.argpartition(vals, -top_k)[-top_k:])
            if len(vals) > top_k:
                order = order[np.argsort(vals[order])[::-1]]
            texts.append(' '.join(str(gene_symbols[cols[j]]) for j in order if vals[j] > 0))
    return texts

def cell_to_text_dense(X_row, gene_symbols, top_k=200):
    row = X_row.toarray().flatten() if sp.issparse(X_row) else np.array(X_row).flatten()
    nz = np.where(row > 0)[0]
    if len(nz) == 0: return ''
    vals = row[nz]
    nz = (nz[np.argsort(vals)[::-1]] if len(nz) <= top_k
          else nz[np.argpartition(vals, -top_k)[-top_k:][np.argsort(vals[np.argpartition(vals, -top_k)[-top_k:]])[::-1]]])
    return ' '.join(gene_symbols[i] for i in nz)

def cell_to_text_robust(h5_path, row_indices, gene_symbols, top_k=200, desc='cells'):
    try:
        return cell_to_text_backed(h5_path, row_indices, gene_symbols, top_k, desc)
    except (KeyError, AttributeError, TypeError, OSError):
        print(f'  [{desc}] fallback to in-memory')
        adata_tmp = sc.read_h5ad(h5_path)
        texts = [cell_to_text_dense(adata_tmp.X[i], gene_symbols, top_k)
                 for i in tqdm(row_indices, desc=f'  {desc}', leave=False)]
        del adata_tmp
        return texts

print('[OK] Utility functions defined')

In [ ]:
print('=' * 60)
print('  Loading training organs')
print('=' * 60)

LABEL_SYNONYM_MAP = {
    'B cells': 'B cell', 'NK cells': 'natural killer cell',
    'Alveolar macrophages': 'alveolar macrophage', 'Mast cells': 'mast cell',
    'Plasma cells': 'plasma cell', 'Classical monocytes': 'classical monocyte',
    'Non-classical monocytes': 'non-classical monocyte',
    'Plasmacytoid DCs': 'plasmacytoid dendritic cell',
    'Smooth muscle': 'smooth muscle cell',
    'CD4 T cells': 'CD4-positive, alpha-beta T cell',
    'CD4-positive helper T cell': 'CD4-positive, alpha-beta T cell',
    'CD8 T cells': 'CD8-positive, alpha-beta T cell',
    'CD4-positive, CD25-positive, alpha-beta regulatory T cell': 'regulatory T cell',
    'CD8-positive, alpha-beta memory T cell, CD45RO-positive': 'CD8-positive, alpha-beta memory T cell',
    'mature alpha-beta T cell': 'alpha-beta T cell',
    'mature NK T cell': 'natural killer T cell',
    'mature B cell': 'B cell',
    'effector memory CD4-positive, alpha-beta T cell, terminally differentiated':
        'effector memory CD4-positive, alpha-beta T cell',
    'dendritic cell, human': 'dendritic cell',
    'group 3 innate lymphoid cell, human': 'group 3 innate lymphoid cell',
    'plasmacytoid dendritic cell, human': 'plasmacytoid dendritic cell',
    'CD14-positive monocyte': 'classical monocyte',
    'CD14-positive, CD16-positive monocyte': 'intermediate monocyte',
    'myeloid dendritic cell': 'conventional dendritic cell',
    'liver dendritic cell': 'conventional dendritic cell',
    'vein endothelial cell': 'endothelial cell of vein',
    'endothelial cell of pericentral hepatic sinusoid': 'endothelial cell of hepatic sinusoid',
    'endothelial cell of periportal hepatic sinusoid': 'endothelial cell of hepatic sinusoid',
    'intrahepatic cholangiocyte': 'cholangiocyte',
    'granulocyte monocyte progenitor cell': 'granulocyte monocyte progenitor',
    'cycling plasma cell': 'plasma cell',
    'inflammatory macrophage': 'macrophage',
    'B_cell': 'B cell', 'B_cell_naive': 'naive B cell',
    'NK_cell': 'natural killer cell', 'plasma_cell': 'plasma cell',
    'plasma_cell_proliferating': 'plasma cell', 'Treg_cell': 'regulatory T cell',
    'alveolar_macrophage': 'alveolar macrophage', 'mast_cell': 'mast cell',
    'monocyte_CSF3R': 'classical monocyte', 'monocyte_SOCS3': 'classical monocyte',
    'monocyte_ITGAL': 'non-classical monocyte', 'monocyte_AREG_EREG': 'monocyte',
    'cDC1': 'conventional dendritic cell',
}

BAD_LABEL_FILTERS = [
    ('liver',      'cell_type', {'malignant cell'}),
    ('lymph_node', 'cell_type', {'stromal cell of pancreas', 'alveolar macrophage'}),
]

loaded_organs = {}
for cfg in ORGAN_CONFIGS:
    adata, obs, valid_idx, gene_syms = load_organ(cfg)
    loaded_organs[cfg['name']] = {'cfg': cfg, 'adata': adata, 'obs': obs,
                                   'valid_idx': valid_idx, 'gene_symbols': gene_syms}

# Disease filter
for name in ['liver', 'lymph_node', 'bone_marrow', 'lymphoid', 'myeloid']:
    d = loaded_organs[name]
    if 'disease' in d['obs'].columns:
        mask = d['obs']['disease'] == 'normal'
        d['obs']       = d['obs'][mask].copy()
        d['valid_idx'] = d['valid_idx'][mask.values]

# Lung pericyte fix
loaded_organs['lung']['obs']['ann_finest_level'] = (
    loaded_organs['lung']['obs']['ann_finest_level'].replace('Pericytes', 'pericyte')
)

# Bad label filter
for organ_name, col, bad_labels in BAD_LABEL_FILTERS:
    d, obs = loaded_organs[organ_name], loaded_organs[organ_name]['obs']
    for lbl in bad_labels:
        n = (obs[col] == lbl).sum()
        if n:
            mask = obs[col] != lbl
            d['obs'] = obs[mask].copy()
            d['valid_idx'] = d['valid_idx'][mask.values]
            obs = d['obs']

# Synonym map
for cfg in ORGAN_CONFIGS:
    name, col = cfg['name'], cfg['label_col']
    obs = loaded_organs[name]['obs']
    for src, tgt in LABEL_SYNONYM_MAP.items():
        n = (obs[col] == src).sum()
        if n:
            loaded_organs[name]['obs'][col] = obs[col].replace(src, tgt)
            obs = loaded_organs[name]['obs']

print('\n[OK] All organs loaded and normalized')

## 4. Load C2S Encoder (Frozen)

In [ ]:
print('Loading C2S encoder ...')
tokenizer = AutoTokenizer.from_pretrained(C2S_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

encoder = AutoModel.from_pretrained(C2S_MODEL_NAME).to(device)
encoder.eval()  # frozen — no dropout, no gradient
for param in encoder.parameters():
    param.requires_grad = False

n_params = sum(p.numel() for p in encoder.parameters()) / 1e6
print(f'  Model   : {C2S_MODEL_NAME}')
print(f'  Params  : {n_params:.1f}M  (all frozen)')
print(f'  Hidden  : {encoder.config.hidden_size}')
print('[OK] Encoder ready')

## 5. Embedding Extraction
*(One forward pass per batch — no gradient tracking)*

In [ ]:
class TextDataset(Dataset):
    """Tokenizes text on-the-fly — avoids storing large token tensors in RAM."""
    def __init__(self, texts, tokenizer, max_length):
        self.texts, self.tokenizer, self.max_length = texts, tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx] if self.texts[idx].strip() else '[PAD]'
        enc  = self.tokenizer(text, truncation=True, max_length=self.max_length,
                              padding='max_length', return_tensors='pt')
        return enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0)


@torch.no_grad()
def extract_embeddings(texts, desc='embedding'):
    """
    Run texts through the frozen C2S encoder and return last-token embeddings.
    Shape: (N, hidden_size)  — same extraction as C2SClassifier.forward().
    """
    dataset = TextDataset(texts, tokenizer, MAX_SEQ_LEN)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
    embeddings = []
    for input_ids, attention_mask in tqdm(loader, desc=f'  {desc}', leave=True):
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        out        = encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state          # (B, T, H)
        seq_len     = attention_mask.sum(dim=1) - 1  # index of last real token
        last_token  = last_hidden[
            torch.arange(last_hidden.size(0), device=device), seq_len
        ]                                            # (B, H)
        embeddings.append(last_token.cpu().float().numpy())
    return np.vstack(embeddings)                     # (N, H)

print('[OK] Embedding function defined')

In [ ]:
print('=' * 60)
print('  Extracting embeddings for all training organs')
print('=' * 60)
t0 = time.time()

CSR_ORGANS = {'lung', 'brain_glia', 'brain_neurons', 'liver', 'lymph_node', 'bone_marrow'}

train_embeddings_list = []
train_labels_list     = []

for cfg in ORGAN_CONFIGS:
    name      = cfg['name']
    d         = loaded_organs[name]
    label_col = cfg['label_col']
    t1 = time.time()
    print(f'\n  [{name}] {len(d["valid_idx"]):,} cells')

    # Text conversion
    texts = (cell_to_text_backed if name in CSR_ORGANS else cell_to_text_robust)(
        cfg['path'], d['valid_idx'], d['gene_symbols'], TOP_K_GENES, desc=f'{name} text'
    )
    labels = d['obs'][label_col].astype(str).values

    # Drop empty texts
    keep   = [bool(t.strip()) for t in texts]
    texts  = [t for t, k in zip(texts,  keep) if k]
    labels = labels[keep]

    # Extract embeddings
    embs = extract_embeddings(texts, desc=f'{name} embed')

    train_embeddings_list.append(embs)
    train_labels_list.append(labels)
    print(f'  [{name}] embedding shape: {embs.shape} | {time.time()-t1:.1f}s')

train_embeddings = np.vstack(train_embeddings_list)    # (N_train, H)
train_labels     = np.concatenate(train_labels_list)   # (N_train,)

print(f'\n  Total embeddings : {train_embeddings.shape}')
print(f'  Unique labels    : {len(np.unique(train_labels))}')
print(f'  Total time       : {(time.time()-t0)/60:.1f} min')
print('[OK] Training embeddings ready')

## 6. Train Logistic Regression
*(L2-regularized, balanced class weights — fits in seconds on CPU)*

In [ ]:
print('=' * 60)
print('  Training Logistic Regression')
print('=' * 60)
t0 = time.time()

# L2-normalize embeddings — helps LR convergence with high-dim inputs
train_embs_norm = normalize(train_embeddings, norm='l2')

# lbfgs uses quasi-Newton updates (second-order) — converges in far fewer
# iterations than first-order solvers (saga/sag) on this problem size.
# verbose=1 prints each iteration number as a lightweight progress signal.
clf = LogisticRegression(
    C=LR_C,
    class_weight='balanced',
    max_iter=LR_MAX_ITER,
    solver='lbfgs',
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)
clf.fit(train_embs_norm, train_labels)

print(f'\n  Classes : {len(clf.classes_)}')
print(f'  Elapsed : {(time.time()-t0)/60:.1f} min')
print('[OK] Logistic regression trained')

## 7. In-Distribution Sanity Check
*(Quick evaluation on a held-out 15% of training data)*

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(train_labels))
idx_train, idx_test = train_test_split(idx, test_size=0.15,
                                        stratify=train_labels, random_state=SEED)

clf_holdout = LogisticRegression(
    C=LR_C,
    class_weight='balanced',
    max_iter=LR_MAX_ITER,
    solver='lbfgs',
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)
clf_holdout.fit(train_embs_norm[idx_train], train_labels[idx_train])

preds_holdout = clf_holdout.predict(train_embs_norm[idx_test])
trues_holdout = train_labels[idx_test]

acc = accuracy_score(trues_holdout, preds_holdout)
unique = np.unique(trues_holdout)
p, r, f, _ = precision_recall_fscore_support(trues_holdout, preds_holdout,
                                              labels=unique, average='macro', zero_division=0)
p_pc, r_pc, f_pc, sup = precision_recall_fscore_support(
    trues_holdout, preds_holdout, labels=unique, zero_division=0
)
per_class_holdout = pd.DataFrame({
    'cell_type': unique, 'precision': p_pc, 'recall': r_pc, 'f1': f_pc, 'support': sup
}).sort_values('recall', ascending=False)

print('=' * 60)
print('  IN-DISTRIBUTION HOLDOUT (15% of training data)')
print('=' * 60)
print(f'  Accuracy      : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Macro recall  : {r:.4f}')
print(f'  Macro F1      : {f:.4f}')
print(f'  0% recall     : {(per_class_holdout["recall"]==0).sum()} classes')
print(f'  >=50% recall  : {(per_class_holdout["recall"]>=0.5).sum()} / {len(per_class_holdout)} classes')
print()
print('  Bottom 10 classes by recall:')
for _, row in per_class_holdout.tail(10).iterrows():
    print(f'    {row["cell_type"]:<50} recall={row["recall"]:.3f}  support={int(row["support"])}')

per_class_holdout.to_csv(os.path.join(OUT_DIR, 'holdout_per_class_metrics.csv'), index=False)
print(f'\n[OK] Holdout evaluation complete')

In [ ]:
pc = per_class_holdout.sort_values('recall')
fig, ax = plt.subplots(figsize=(10, max(4, len(pc) * 0.28)))
colors = ['#d73027' if r < 0.5 else '#fee090' if r < 0.8 else '#1a9850' for r in pc['recall']]
ax.barh(pc['cell_type'], pc['recall'], color=colors)
ax.axvline(0.5, color='red',    ls='--', lw=1.2, label='50%')
ax.axvline(0.8, color='orange', ls='--', lw=1.2, label='80%')
ax.set(xlabel='Recall', title=f'Per-class Recall — Holdout (LR baseline)\nMacro recall={r:.3f}', xlim=[0, 1])
ax.legend(fontsize=9); ax.tick_params(axis='y', labelsize=7); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'holdout_recall.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Zero-Shot Inference on Lab Datasets
*(Refit final classifier on all training data, then predict lab cells)*

In [ ]:
# Use the full-data classifier (clf) for lab inference
print('=' * 60)
print('  Zero-shot inference on lab datasets')
print('=' * 60)

lab_results = {}

for lab_cfg in LAB_CONFIGS:
    lab_name  = lab_cfg['name']
    label_col = lab_cfg['label_col']
    print(f'\n  --- {lab_name} ---')
    t0 = time.time()

    adata_lab    = sc.read_h5ad(lab_cfg['path'])
    lab_gene_sym = get_gene_symbols(adata_lab)
    true_labels  = adata_lab.obs[label_col].astype(str).values

    # Text conversion
    X_lab = adata_lab.X
    texts = [
        cell_to_text_dense(X_lab[i], lab_gene_sym, TOP_K_GENES)
        for i in tqdm(range(adata_lab.n_obs), desc='    text', leave=False)
    ]

    # Embed
    embs_lab = extract_embeddings(texts, desc=f'{lab_name} embed')
    embs_lab_norm = normalize(embs_lab, norm='l2')

    # Predict
    pred_labels = clf.predict(embs_lab_norm)
    pred_probs  = clf.predict_proba(embs_lab_norm)   # (N, n_classes)
    confidences = pred_probs.max(axis=1)

    df = pd.DataFrame({
        'true_label': true_labels,
        'pred_label': pred_labels,
        'confidence': confidences,
    })
    safe_name = lab_name.replace(' ', '_').replace('(', '').replace(')', '')
    df.to_csv(os.path.join(OUT_DIR, f'lab_{safe_name}_predictions.csv'), index=False)
    lab_results[lab_name] = df

    breakdown = (
        df.groupby(['true_label', 'pred_label'])
        .size().reset_index(name='count')
    )
    breakdown['pct'] = breakdown.groupby('true_label')['count'].transform(lambda x: x / x.sum())
    breakdown.to_csv(os.path.join(OUT_DIR, f'lab_{safe_name}_breakdown.csv'), index=False)

    print(f'    Cells: {len(df):,}  |  Mean confidence: {confidences.mean():.4f}  |  {time.time()-t0:.1f}s')
    print(f'    Top predictions:')
    for true_lbl in sorted(df['true_label'].unique()):
        sub = breakdown[breakdown['true_label'] == true_lbl].sort_values('pct', ascending=False)
        n_total = sub['count'].sum()
        top = sub.iloc[0]
        print(f'      {true_lbl:<45} -> {top["pred_label"]:<40} {top["pct"]*100:.1f}%  (n={n_total})')

print('\n[OK] Zero-shot inference complete')

## 9. Zero-Shot Visualizations

In [ ]:
def plot_stacked_bar(df, lab_name):
    breakdown = (
        df.groupby(['true_label', 'pred_label'])
        .size().reset_index(name='count')
    )
    breakdown['pct'] = breakdown.groupby('true_label')['count'].transform(lambda x: x / x.sum())
    unique_true  = sorted(df['true_label'].unique())
    active_preds = df['pred_label'].value_counts().index.tolist()
    pivot = breakdown.pivot_table(
        index='true_label', columns='pred_label', values='pct', fill_value=0
    ).reindex(index=sorted(breakdown['true_label'].unique()), fill_value=0)

    palette   = plt.cm.get_cmap('tab20', len(active_preds))
    color_map = {c: palette(i) for i, c in enumerate(active_preds)}

    fig, ax = plt.subplots(figsize=(max(8, len(unique_true) * 0.9), 6))
    bottoms = np.zeros(len(pivot))
    x = np.arange(len(pivot))
    for pred_cls in active_preds:
        if pred_cls not in pivot.columns: continue
        vals = pivot[pred_cls].values
        ax.bar(x, vals, bottom=bottoms, label=pred_cls,
               color=color_map[pred_cls], edgecolor='white', linewidth=0.3)
        bottoms += vals
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Fraction of cells')
    ax.set_title(f'LR Baseline — prediction mix: {lab_name}')
    ax.set_ylim(0, 1.05)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7,
              title='Predicted', title_fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    safe = lab_name.replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(os.path.join(OUT_DIR, f'lab_{safe}_stacked_bar.png'), dpi=150, bbox_inches='tight')
    plt.show()


def plot_confidence(df, lab_name):
    unique_true = sorted(df['true_label'].unique())
    n_cols = min(4, len(unique_true))
    n_rows = int(np.ceil(len(unique_true) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 2.8))
    axes = np.array(axes).flatten()
    for i, true_lbl in enumerate(unique_true):
        conf = df[df['true_label'] == true_lbl]['confidence'].values
        axes[i].hist(conf, bins=20, range=(0, 1), color='steelblue',
                     edgecolor='white', linewidth=0.4)
        axes[i].axvline(conf.mean(), color='red', ls='--', lw=1,
                        label=f'mean={conf.mean():.2f}')
        axes[i].set_title(true_lbl, fontsize=8)
        axes[i].set_xlim(0, 1)
        axes[i].legend(fontsize=7)
        axes[i].tick_params(labelsize=7)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    fig.suptitle(f'LR Baseline confidence — {lab_name}', fontsize=11, y=1.01)
    plt.tight_layout()
    safe = lab_name.replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(os.path.join(OUT_DIR, f'lab_{safe}_confidence.png'), dpi=150, bbox_inches='tight')
    plt.show()


for lab_name, df in lab_results.items():
    print(f'\nPlotting: {lab_name}')
    plot_stacked_bar(df, lab_name)
    plot_confidence(df, lab_name)

## 10. Summary

In [ ]:
print('=' * 60)
print('  C2S FROZEN EMBEDDING + LOGISTIC REGRESSION — SUMMARY')
print('=' * 60)

print('\nTRAINING DATA')
for cfg in ORGAN_CONFIGS:
    name, col = cfg['name'], cfg['label_col']
    n = len(loaded_organs[name]['obs'])
    t = loaded_organs[name]['obs'][col].nunique()
    print(f'  {name:<22} {n:>7,} cells  {t:>4} types')
print(f'  Total training embeddings : {len(train_labels):,}')
print(f'  LR classes                : {len(clf.classes_)}')

print('\nIN-DISTRIBUTION HOLDOUT (15%)')
print(f'  Accuracy     : {acc:.4f}')
print(f'  Macro recall : {r:.4f}')
print(f'  Macro F1     : {f:.4f}')
print(f'  0% recall    : {(per_class_holdout["recall"]==0).sum()} / {len(per_class_holdout)} classes')

print('\nZERO-SHOT LAB VALIDATION (dominant prediction per true label)')
for lab_name, df in lab_results.items():
    print(f'\n  {lab_name}  (mean confidence: {df["confidence"].mean():.4f})')
    breakdown = (
        df.groupby(['true_label', 'pred_label'])
        .size().reset_index(name='count')
    )
    breakdown['pct'] = breakdown.groupby('true_label')['count'].transform(
        lambda x: x / x.sum()
    )
    for true_lbl in sorted(df['true_label'].unique()):
        sub = breakdown[breakdown['true_label'] == true_lbl].sort_values('pct', ascending=False)
        top = sub.iloc[0]
        print(f'    {true_lbl:<45} -> {top["pred_label"]:<40} {top["pct"]*100:.1f}%')

print(f'\n[OK] All results saved to {OUT_DIR}/')